In [4]:
# ==============================================================================
import pandas as pd
import numpy as np

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

# Preprocesado y modelado
# ==============================================================================
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error,accuracy_score,mean_absolute_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
import xgboost as xgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from imblearn.over_sampling import SMOTE,SVMSMOTE
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.tree import DecisionTreeClassifier
# ======================================================================================
import joblib
from sklearn import metrics
from sklearn.cluster import SpectralClustering, KMeans, DBSCAN, AgglomerativeClustering
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import ElasticNet
import statistics
import numpy as np
from sklearn.metrics import pairwise_distances_argmin_min
from scipy.spatial.distance import cdist
import joblib
from sklearn.metrics import silhouette_score



In [5]:
'''
    Esta clase permite realizar la codificación dummy especificameente variables categoricas
'''
class OneHotCoding():
    def __init__(self, df, bin_features):
        self.bin_features = bin_features
        self.df = df

    # Metodo para realizar la codificacion dummy a las variables categoricas

    def dummyCodification(self):
        cat_features = self.df.select_dtypes(include = ["object", "category"]).columns
        bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})
        categorical_features = [x for x in cat_features if x not in self.bin_features]
        df_cat = pd.get_dummies(self.df[categorical_features], dtype=int)
        self.df.drop(cat_features, axis = 1, inplace = True)
        df_final = pd.concat([self.df,df_cat,bin_dataset ], axis = 1)
        df_final.to_csv("daset_codificado.csv")
        print("Ejeción Terminada")
        return df_final

'''

    Esta clase permite Calcular modelo de regresión Elasticnet por cada grupo
'''
class LinearRegession():
    def __init__(self, df, alpha, l1_ratio):
        self.df = df
        self.alpha = alpha
        self.l1_ratio = l1_ratio


    def CalcularModeloLR(self):
        # alpha=0.1, l1_ratio=0.97
        Y = self.df.RDT_AJUSTADO.values
        X = self.df.drop(["RDT_AJUSTADO","Grupo"], axis=1).values
        modelElasticNet = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, random_state=123)
        model = modelElasticNet.fit(X,Y)
        #r_2 = model.score(X,Y)
        # Pedicciones
        yhat = model.predict(X)
        r_2 = r2_score(Y, yhat)


        return [r_2,model]

'''
  Función para evaluar el desempeño de los modelos de regresión
'''
def metricasModelosRegresion(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # También puedes crear un DataFrame para mostrar las métricas juntas
    metrics_df = pd.DataFrame({
        'Métrica': ['MSE', 'RMSE', 'MAE', 'R²'],
        'Valor': [mse, rmse, mae, r2]
    })
    return r2 , metrics_df



'''
  Función para calcular regresion lineal , teniendo en cuenta
  libreria de  Stasts models.
'''

def LinearRegessionOLMS(df_entrenamiento):
    Y = df_entrenamiento.RDT_AJUSTADO.values
    X = df_entrenamiento.drop(["RDT_AJUSTADO","Grupo"], axis=1).values
    # Agreago Constantes al modelo
    X_train = sm.add_constant(X, prepend=True)
    model = sm.OLS(Y, X_train)
    res= model.fit_regularized(method='elastic_net',alpha=0.1, L1_wt=0.97)
    model_fit_regularized = model.fit(params=res.params)
    r_2 = model_fit_regularized.rsquared
    #print("Nuevo r_2: ", r_2)
    return r_2, model_fit_regularized

'''
  Función para calcular correlaciones iniciales para cada uno de los grupos.
'''
def CalcularCorrelationInitialOLS(gruposDefinitivos):
    correlacionesIniciales=[]
    lista_modelos = []
    for i in range(len(gruposDefinitivos)):
        #r_2, model = LinearRegessionOLMS(gruposDefinitivos[i])
        r_2, model = LinearRegession(gruposDefinitivos[i],alpha=0.1,l1_ratio=0.97).CalcularModeloLR()
        # Guardo Correlacion
        correlacionesIniciales.append(r_2)
        lista_modelos.append(model)
        # Gurado modelos
        #name_model = f'models/modelo_ols_{i}.pkl'
        #joblib.dump(model,name_model) # Guardo el modelo.
    return correlacionesIniciales,lista_modelos


def cargarDatsetUnido():
    g0 = pd.read_excel("../DataOpt/grupo_N0.xlsx").drop(["Unnamed: 0"], axis=1)
    g1 = pd.read_excel("../DataOpt/grupo_N1.xlsx").drop(["Unnamed: 0"], axis=1)
    g2 = pd.read_excel("../DataOpt/grupo_N2.xlsx").drop(["Unnamed: 0"], axis=1)
    print(f"Longitud G0: ",{len(g0)},"longitud G1: ", {len(g1)} , "Longitud G2: ", {len(g2)})
    df_group = pd.concat([g0,g1,g2],axis=0)
    print(df_group.shape)
    return df_group



'''
  Función para realizar balanceo de clases de c/u de los dataframes.
'''
def SmoteDataSetDesbalanceado(df):
    y= df.Grupo
    X= df.drop(["Grupo"],axis=1)
    x_resampled, y_resampeld =  SMOTE(sampling_strategy='not majority',random_state=42).fit_resample(X, y)
    df_smote = pd.concat([x_resampled, y_resampeld],axis=1)
    return df_smote

'''
  Función para dividir c/d grupo en un unico dataset
'''
def calcularGruposDefinitivos(dataset):
    ListagruposDefinitivos=[]
    for i in range(len(dataset.Grupo.unique())):
        filtro_grupo = dataset[dataset.Grupo==i]
        ListagruposDefinitivos.append(filtro_grupo)

    return ListagruposDefinitivos



def predictionYield(x_dataset_test,lista_modelos,cluster_asignado):
        y_pred_list = []
        for z in range(len(x_dataset_test)):
            #print(f"Registro {z}, cluster asignado {cluster_asignado[z]}")
            y_pred = lista_modelos[cluster_asignado[z]].predict(x_dataset_test.values[z].reshape(1,-1))
            y_pred_list.append(y_pred[0])
        return y_pred_list


'''
   Algortimos de clustering para realziar la separación inicial de los grupos.
'''
def AlgortimoClustering(dataset_train_clsuter, alg, semilla):
  if alg==1:
    gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=semilla).fit(dataset_train_cluster)
    lables =  gm.predict(dataset_train_cluster)
  if alg==2:
    sc = SpectralClustering(n_clusters=2,assign_labels='discretize',random_state=semilla).fit(dataset_train_cluster)
    lables = sc.labels_
  if alg==3:
    km = KMeans(n_clusters=2, random_state=semilla, n_init=10).fit(dataset_train_cluster)
    lables = km.labels_

  return lables







'''
  Funcion para aplicar modelo de clasificación DT, al dataset etiquedado.
  -
'''

def etapaClasfDT(df_tag):
  y = df_tag.Grupo
  X = df_tag.drop(["Grupo","RDT_AJUSTADO"],axis=1)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234, stratify=y)
  clf = DecisionTreeClassifier()
  clf.fit(X_train,y_train)
  y_pred = clf.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  print(f"Precisión del modelo clasificación: {accuracy * 100:.2f}%")
  return clf



In [61]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("DatasetFinalFP.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)



dataset_train_cluster = d_train_x
gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=15).fit(dataset_train_cluster)
target =  gm.predict(dataset_train_cluster)
dataset_tag = dataset_features.loc[lista_indices_train_p1]
dataset_tag["Grupo"] =target

grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
print("Grupos Definitivos: ", len(grupos_definitivos))
# Calculos correlaciones OLMS y modelos
lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
print("lista Correlaciones Iniciales: ", lista_corr_olsms)
# Etapa de Clasificación
model_clasf = etapaClasfDT(dataset_tag)
# Se valida con lo que nunca se vio.
grupo_asignado = model_clasf.predict(d_test_x)
y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
r2, mdf= metricasModelosRegresion(d_test_y, y_pred_test)
print("r2: ", r2)


# Con conjunto datos jamas visto  (Conjunto Validación)
# =======================================================
grupo_asignado = model_clasf.predict(x_dataset_test)
y_pred_val = predictionYield(x_dataset_test,models,grupo_asignado)
r2_new, metrics  = metricasModelosRegresion(y_dataset_test, y_pred_val)

print("r2_new: ", r2_new)


<ipython-input-5-fa4340507517>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8556693643700476, 0.7417871863817992]
Precisión del modelo clasificación: 96.15%
r2:  0.7217598636784903
r2_new:  0.7802646350313625


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.566e+07, tolerance: 3.558e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.860e+06, tolerance: 8.203e+04
  model = cd_fast.enet_coordinate_descent(


In [50]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("DatasetFinalFP.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING (Inicialización de la semilla y mejor r2)
# ==============================================================
semilla = 0
best_r2 = 0

lista_r2 = []
for r in range(30):
  dataset_train_cluster = d_train_x
  gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=r).fit(dataset_train_cluster)
  target =  gm.predict(dataset_train_cluster)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2, m_= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 >= best_r2:
    best_model_clasf = model_clasf
    best_models_reg = models
    best_r2 = r2
    semilla = r


# Con conjunto datos jamas visto  (Conjunto Validación)
# =======================================================
grupo_asignado = best_model_clasf.predict(x_dataset_test)
y_pred_val = predictionYield(x_dataset_test,best_models_reg,grupo_asignado)
r2_new, mdf_= metricasModelosRegresion(y_dataset_test, y_pred_val)


print("-------Mejor R2: ", best_r2)
print("-------Semilla Cluster: ", semilla)
print("-------Dataset de test: r2_new: ", r2_new)

<ipython-input-5-fa4340507517>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8556693643700476, 0.7417871863817992]


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.566e+07, tolerance: 3.558e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.860e+06, tolerance: 8.203e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 94.62%
r2:  0.7217598636784903
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 94.62%
r2:  0.7214304231774126


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


...

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 93.85%


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


r2:  0.7214304231774126
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 94.62%
r2:  0.7214304231774126
-------Mejor R2:  0.7217598636784903
-------Semilla Cluster:  15
-------Dataset de test: r2_new:  0.7823803157909871


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


In [52]:
# Metricas modelos de regresión
# ==================================================
metrics

,Métrica,Valor
0,MSE,591674.795007
1,RMSE,769.204001
2,MAE,585.949838
3,R²,0.729556


In [ ]:
dataset_tag

,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,...,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,Grupo
51,5,47,72,75500,17,0,0,0,0,0,...,1,0,0,0,1,0,0,0,0,0
326,5,46,79,73000,5,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,0
682,5,53,85,60000,10,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
752,4,47,82,55000,10,1,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
678,6,52,94,60000,11,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,6,49,85,72000,16,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,0
141,5,48,75,71000,7,0,0,0,0,0,...,1,1,0,0,1,1,0,1,0,0
731,4,47,82,62000,8,0,0,0,0,0,...,0,0,0,0,1,0,0,1,1,1
146,5,51,80,60000,25,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


In [ ]:
dataset_tag[dataset_tag["Grupo"]==1]

,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,...,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,Grupo
752,4,47,82,55000,10,1,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
120,4,47,87,55000,4,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
226,3,47,81,55000,6,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
251,4,49,85,60000,6,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
284,4,49,82,65000,6,0,0,0,0,0,...,1,0,0,0,1,0,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,6,56,72,60000,10,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,1
203,4,47,76,68000,6,0,0,0,0,0,...,1,0,0,0,1,0,0,1,1,1
782,3,47,79,60000,4,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1
767,4,47,85,55000,6,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,1


In [ ]:
new_feature_importance = ['Rhum_Avg_For',
 'Sol_Ener_Accu_Veg',
 'ContMalMec_Emer_Flor',
 'Rain_Accu_Veg',
 'Rain_10_Freq_Veg',
 'Rhum_Avg_Veg',
 'Temp_Max_34_Freq_For',
 'Diurnal_Range_Avg_Veg',
 'POSICION_PERFIL_RASTA_PLANO',
 'Rain_Accu_For',
 'Grupo']
dataset_sample = dataset_tag[new_feature_importance].reset_index(drop=True)
dataset_sample

,Rhum_Avg_For,Sol_Ener_Accu_Veg,ContMalMec_Emer_Flor,Rain_Accu_Veg,Rain_10_Freq_Veg,Rhum_Avg_Veg,Temp_Max_34_Freq_For,Diurnal_Range_Avg_Veg,POSICION_PERFIL_RASTA_PLANO,Rain_Accu_For,Grupo
0,81.13,15623.60,0,98.2,0.05,77.74,0.40,9.53,1,322.7,0
1,79.04,15197.96,0,412.1,0.25,80.33,0.26,8.63,1,216.1,0
2,78.65,14696.52,0,198.3,0.15,80.80,0.60,8.91,1,227.2,0
3,79.79,14410.85,0,268.6,0.18,82.55,0.19,7.09,1,109.7,1
4,79.01,15178.36,0,189.1,0.15,79.70,0.50,9.17,1,259.6,0
...,...,...,...,...,...,...,...,...,...,...,...
642,78.60,15123.25,0,426.4,0.28,79.97,0.24,8.88,1,220.9,0
643,80.19,14575.71,0,249.0,0.15,82.25,0.50,8.70,1,129.3,0
644,83.34,16491.43,0,337.9,0.22,83.93,0.16,9.08,1,190.2,1
645,78.65,15307.87,0,188.0,0.15,80.90,0.60,8.90,1,255.9,0


In [ ]:
probabilidades = gm.predict_proba(dataset_train_cluster)
df_pertenencia = pd.DataFrame(probabilidades, columns=['Cluster 0', 'Cluster 1'])
#df_pertenencia.to_csv("probabilidades.csv")
df_pertenencia

,Cluster 0,Cluster 1
0,1.000000e+00,0.000000e+00
1,1.000000e+00,3.983553e-72
2,1.000000e+00,5.711099e-91
3,2.410406e-237,1.000000e+00
4,1.000000e+00,3.260695e-296
...,...,...
642,1.000000e+00,6.509377e-181
643,1.000000e+00,0.000000e+00
644,1.035402e-61,1.000000e+00
645,1.000000e+00,0.000000e+00


In [ ]:
dataset_pruebas = pd.concat([dataset_sample, df_pertenencia], axis=1)
dataset_pruebas

,Rhum_Avg_For,Sol_Ener_Accu_Veg,ContMalMec_Emer_Flor,Rain_Accu_Veg,Rain_10_Freq_Veg,Rhum_Avg_Veg,Temp_Max_34_Freq_For,Diurnal_Range_Avg_Veg,POSICION_PERFIL_RASTA_PLANO,Rain_Accu_For,Grupo,Cluster 0,Cluster 1
0,81.13,15623.60,0,98.2,0.05,77.74,0.40,9.53,1,322.7,0,1.000000e+00,0.000000e+00
1,79.04,15197.96,0,412.1,0.25,80.33,0.26,8.63,1,216.1,0,1.000000e+00,3.983553e-72
2,78.65,14696.52,0,198.3,0.15,80.80,0.60,8.91,1,227.2,0,1.000000e+00,5.711099e-91
3,79.79,14410.85,0,268.6,0.18,82.55,0.19,7.09,1,109.7,1,2.410406e-237,1.000000e+00
4,79.01,15178.36,0,189.1,0.15,79.70,0.50,9.17,1,259.6,0,1.000000e+00,3.260695e-296
...,...,...,...,...,...,...,...,...,...,...,...,...,...
642,78.60,15123.25,0,426.4,0.28,79.97,0.24,8.88,1,220.9,0,1.000000e+00,6.509377e-181
643,80.19,14575.71,0,249.0,0.15,82.25,0.50,8.70,1,129.3,0,1.000000e+00,0.000000e+00
644,83.34,16491.43,0,337.9,0.22,83.93,0.16,9.08,1,190.2,1,1.035402e-61,1.000000e+00
645,78.65,15307.87,0,188.0,0.15,80.90,0.60,8.90,1,255.9,0,1.000000e+00,0.000000e+00


In [ ]:
# Proceso Cluster  0
# ====================================================================
cluster0 = dataset_pruebas[dataset_pruebas["Grupo"]==0]
suma_pertenencia_c0 = sum(cluster0["Cluster 0"]) * 100
centroide_c0 = cluster0.sum()[0:-3]/suma_pertenencia_c0
print("La suma de pertenencia del cluster 1 es:  " ,suma_pertenencia_c0)
cluster0

La suma de pertenencia del cluster 1 es:   43200.0


,Rhum_Avg_For,Sol_Ener_Accu_Veg,ContMalMec_Emer_Flor,Rain_Accu_Veg,Rain_10_Freq_Veg,Rhum_Avg_Veg,Temp_Max_34_Freq_For,Diurnal_Range_Avg_Veg,POSICION_PERFIL_RASTA_PLANO,Rain_Accu_For,Grupo,Cluster 0,Cluster 1
0,81.13,15623.60,0,98.2,0.05,77.74,0.40,9.53,1,322.7,0,1.0,0.000000e+00
1,79.04,15197.96,0,412.1,0.25,80.33,0.26,8.63,1,216.1,0,1.0,3.983553e-72
2,78.65,14696.52,0,198.3,0.15,80.80,0.60,8.91,1,227.2,0,1.0,5.711099e-91
4,79.01,15178.36,0,189.1,0.15,79.70,0.50,9.17,1,259.6,0,1.0,3.260695e-296
5,78.69,14658.39,0,198.3,0.15,80.70,0.60,8.96,1,227.2,0,1.0,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
639,80.19,13566.84,0,167.3,0.12,82.08,0.05,8.09,1,142.0,0,1.0,0.000000e+00
642,78.60,15123.25,0,426.4,0.28,79.97,0.24,8.88,1,220.9,0,1.0,6.509377e-181
643,80.19,14575.71,0,249.0,0.15,82.25,0.50,8.70,1,129.3,0,1.0,0.000000e+00
645,78.65,15307.87,0,188.0,0.15,80.90,0.60,8.90,1,255.9,0,1.0,0.000000e+00


In [ ]:
centroide_c0

,0
Rhum_Avg_For,0.798130
Sol_Ener_Accu_Veg,152.251494
ContMalMec_Emer_Flor,0.000023
Rain_Accu_Veg,2.382051
Rain_10_Freq_Veg,0.001588
Rhum_Avg_Veg,0.801153
Temp_Max_34_Freq_For,0.003831
Diurnal_Range_Avg_Veg,0.089813
POSICION_PERFIL_RASTA_PLANO,0.009514
Rain_Accu_For,2.597860


In [ ]:
#
# # Proceso Cluster  1
# ====================================================================
cluster1 = dataset_pruebas[dataset_pruebas["Grupo"]==1]
suma_pertenencia_c1= sum(cluster1["Cluster 1"]) * 100
centroide_c1 = cluster1.sum()[0:-3]/suma_pertenencia_c1
print("La suma de pertenencia del cluster 1 es:  " ,suma_pertenencia_c1)
cluster1

La suma de pertenencia del cluster 1 es:   21450.80140829417


,Rhum_Avg_For,Sol_Ener_Accu_Veg,ContMalMec_Emer_Flor,Rain_Accu_Veg,Rain_10_Freq_Veg,Rhum_Avg_Veg,Temp_Max_34_Freq_For,Diurnal_Range_Avg_Veg,POSICION_PERFIL_RASTA_PLANO,Rain_Accu_For,Grupo,Cluster 0,Cluster 1
3,79.79,14410.85,0,268.6,0.18,82.55,0.19,7.09,1,109.7,1,2.410406e-237,1.0
7,84.63,14550.35,0,174.0,0.18,88.88,0.19,7.16,1,63.8,1,2.139253e-282,1.0
11,86.62,16208.79,0,99.3,0.10,80.08,0.21,10.67,1,403.5,1,1.243361e-115,1.0
12,84.16,14057.88,0,161.0,0.15,89.03,0.19,7.12,1,163.8,1,7.299605e-197,1.0
13,83.27,16264.00,0,315.5,0.20,84.26,0.19,8.97,1,212.2,1,1.620757e-131,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,80.40,14523.07,0,248.6,0.12,81.42,0.21,7.75,1,227.2,1,5.269746e-146,1.0
636,83.21,16049.11,0,266.6,0.20,84.56,0.21,8.84,1,293.9,1,6.517123e-123,1.0
640,80.52,15588.59,0,104.6,0.05,78.77,0.45,9.23,1,306.6,1,2.327748e-85,1.0
641,85.48,14347.75,0,234.7,0.20,81.30,0.38,9.57,1,333.7,1,4.777284e-121,1.0


In [ ]:
centroide_c1

,0
Rhum_Avg_For,0.828864
Sol_Ener_Accu_Veg,152.144053
ContMalMec_Emer_Flor,0.000140
Rain_Accu_Veg,2.175578
Rain_10_Freq_Veg,0.001615
Rhum_Avg_Veg,0.835966
Temp_Max_34_Freq_For,0.002566
Diurnal_Range_Avg_Veg,0.085463
POSICION_PERFIL_RASTA_PLANO,0.009790
Rain_Accu_For,2.314492


In [ ]:
# Definición de grupos
# ========================================================
grupos_definitivos[0].to_csv("grupo_0.csv")
grupos_definitivos[1].to_csv("grupo_1.csv")


#### Feature importance del modelo

In [ ]:

def calcularImportanciaFeatures(grupos_definitivos):
    # Lista para almacenar la importancia de cada feature en cada modelo
    feature_names = grupos_definitivos[0].drop(["RDT_AJUSTADO", "Grupo"], axis=1).columns.tolist()

    coeficientes_modelo = { "Feature": feature_names }  # Diccionario para almacenar los coeficientes

    for g in range(len(grupos_definitivos)):
        Y_ = grupos_definitivos[g].RDT_AJUSTADO
        X_ = grupos_definitivos[g].drop(["RDT_AJUSTADO", "Grupo"], axis=1)

        # Normalizar
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X_)

        # Entrenar modelo ElasticNet
        elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.97)
        elastic_net.fit(X_scaled, Y_)

        # Obtener coeficientes absolutos
        coef_norm = abs(elastic_net.coef_)

        # Guardar en el diccionario con clave "Modelo 1", "Modelo 2", etc.
        coeficientes_modelo[f"Coef Modelo {g+1}"] = coef_norm

    # Convertir el diccionario en un DataFrame
    df_importancia = pd.DataFrame(coeficientes_modelo)

    return df_importancia , feature_names, scaler

# Llamada a la función (suponiendo que grupos_definitivos es una lista de DataFrames)
df_importancia , features_names , scaler = calcularImportanciaFeatures(grupos_definitivos)
df_importancia


,Feature,Coef Modelo 1,Coef Modelo 2
0,DIAS_EN_EMERGER,389.579226,160.652468
1,DIAS_EN_EMERGER_A_FLORECER,179.588005,4.716670
2,DIAS_EN_FLORECER_A_COSECHAR,136.904986,36.467368
3,POBLACION_20DIAS_AJT,663.006783,6.167489
4,ALTURA_LOT,308.850037,270.837661
...,...,...,...
169,OBSERVA_RAICES_VIVAS_RASTA,661.862861,32.645376
170,OBSERVA_HOJARASCA_MO_RASTA,65.402419,150.497036
171,SUELO_NEGRO_BLANDO_RASTA,250.623061,0.000000
172,CUCHILLO_PRIMER_HTE_RASTA,194.915158,33.989042


In [ ]:
len(features_names)


174

In [ ]:
# Ordeno FI modelo 1 de mayor a menor
df1 = df_importancia[["Feature", "Coef Modelo 1"]].sort_values(by="Coef Modelo 1", ascending=False).reset_index(drop=True)
#df1.to_excel("df1.xlsx")
df1

,Feature,Coef Modelo 1
0,Sol_Ener_Accu_Veg,1264.730210
1,Rhum_Avg_For,1166.330810
2,Rain_Accu_Veg,1119.057339
3,PROFUND_RAICES_VIVAS_RASTA,985.015780
4,MATERIAL_GENETICO_Cerato (Syngenta),750.308673
...,...,...
169,TERRENO_CIRCUN_RASTA_ONDULADO,0.000000
170,CULT_ANT_Yuca,0.000000
171,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_SIN COBERTURA,0.000000
172,d.interno_EXCESIVO,0.000000


In [ ]:
# Ordeno FI modelo 2 de mayor a menor
df2 = df_importancia[["Feature", "Coef Modelo 2"]].sort_values(by="Coef Modelo 2", ascending=False).reset_index(drop=True)
#df2.to_excel("df2.xlsx")
df2


,Feature,Coef Modelo 2
0,ContPlaQui_Flor_Cose,654.860435
1,Rhum_Avg_Veg,634.373832
2,MATERIAL_GENETICO_PAC 105,630.626545
3,Diurnal_Range_Avg_Veg,624.997211
4,Rhum_Avg_For,609.803120
...,...,...
169,ESTRUCTURA_RASTA_SUELTA O POLVOSA,0.000000
170,d.interno_EXCESIVO,0.000000
171,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_SIN COBERTURA,0.000000
172,OBSERVA_EROSION_RASTA,0.000000


In [ ]:
# Obtengo las caracteristicas mas importantes en ambos modelo.
df1["Pos_df2"] = df1["Feature"].apply(lambda x: df2[df2["Feature"] == x].index[0])
df_importance = df1.reset_index(drop=False).rename(columns={"index": "Pos_df1"})
df_importance

,Pos_df1,Feature,Coef Modelo 1,Pos_df2
0,0,Sol_Ener_Accu_Veg,1264.730210,20
1,1,Rhum_Avg_For,1166.330810,4
2,2,Rain_Accu_Veg,1119.057339,21
3,3,PROFUND_RAICES_VIVAS_RASTA,985.015780,87
4,4,MATERIAL_GENETICO_Cerato (Syngenta),750.308673,156
...,...,...,...,...
169,169,TERRENO_CIRCUN_RASTA_ONDULADO,0.000000,164
170,170,CULT_ANT_Yuca,0.000000,102
171,171,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_SIN COBERTURA,0.000000,171
172,172,d.interno_EXCESIVO,0.000000,170


In [ ]:
df_importance["Sum_positions"] = df_importance["Pos_df1"] + df_importance["Pos_df2"]
df_importance = df_importance.sort_values(by="Sum_positions").reset_index(drop=True)
df_importance

,Pos_df1,Feature,Coef Modelo 1,Pos_df2,Sum_positions
0,1,Rhum_Avg_For,1166.330810,4,5
1,0,Sol_Ener_Accu_Veg,1264.730210,20,20
2,7,ContMalMec_Emer_Flor,700.079822,15,22
3,2,Rain_Accu_Veg,1119.057339,21,23
4,14,Rain_10_Freq_Veg,633.941998,9,23
...,...,...,...,...,...
169,165,MATERIAL_GENETICO_Status (Syngenta),0.000000,166,331
170,169,TERRENO_CIRCUN_RASTA_ONDULADO,0.000000,164,333
171,166,ESTRUCTURA_RASTA_SUELTA O POLVOSA,0.000000,169,335
172,171,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_SIN COBERTURA,0.000000,171,342


In [ ]:
# Obtengo las caracteristicas mas importantes para ambos modelos
# ===============================================================
df_importance.to_excel("df_importance.xlsx")

In [ ]:
df_importance.Feature[0:20]

,Feature
0,Rhum_Avg_For
1,Sol_Ener_Accu_Veg
2,ContMalMec_Emer_Flor
3,Rain_Accu_Veg
4,Rain_10_Freq_Veg
5,Rhum_Avg_Veg
6,Temp_Max_34_Freq_For
7,Diurnal_Range_Avg_Veg
8,POSICION_PERFIL_RASTA_PLANO
9,Rain_Accu_For


In [ ]:
# Obtengo las 10 caracteristicas mas importantes Que comparaten ambos cluster
# de acuerdo a los coeficientes..
# ======================================================================================

features_analize = list(df_importance.Feature[:10].values)
features_analize

['Rhum_Avg_For',
 'Sol_Ener_Accu_Veg',
 'ContMalMec_Emer_Flor',
 'Rain_Accu_Veg',
 'Rain_10_Freq_Veg',
 'Rhum_Avg_Veg',
 'Temp_Max_34_Freq_For',
 'Diurnal_Range_Avg_Veg',
 'POSICION_PERFIL_RASTA_PLANO',
 'Rain_Accu_For']

### Analisis centroides del algortimo de Mezclas Gausianas

In [ ]:
# Obtengo las medias de cada algortimo
# =================================================================================
centroides = gm.means_
centroides

array([[0.35827854, 0.36567926, 0.50997762, 0.7372474 , 0.16046532,
        0.06004812, 0.02078589, 0.        , 0.00230954, 0.00230954,
        0.07967923, 0.17860465, 0.24307137, 0.01963111, 0.0646672 ,
        0.06582197, 0.20477234, 0.040417  , 0.00740058, 0.03257855,
        0.35598024, 0.00230954, 0.0524988 , 0.03465287, 0.00323336,
        0.0338502 , 0.14018156, 0.00461909, 0.00692863, 0.06851644,
        0.40581052, 0.32967656, 0.37259556, 0.45171451, 0.0618961 ,
        0.01855253, 0.09053408, 0.4686011 , 0.60061909, 0.        ,
        0.24871814, 0.01743551, 0.02842909, 0.10137558, 0.0411646 ,
        0.06665778, 0.43742327, 0.0604218 , 0.0049693 , 0.03659494,
        0.14837822, 0.00329918, 0.37763105, 0.38828059, 0.0059567 ,
        0.03977102, 0.00592289, 0.48760206, 0.5937731 , 0.54444378,
        0.42076804, 0.5467012 , 0.47730123, 0.49780022, 0.4725428 ,
        0.36045972, 0.48596601, 0.58364479, 0.60511518, 0.46689048,
        0.48549584, 0.49152592, 0.5848493 , 0.69

In [ ]:
#  Desnormalizo los centroides
centroides_norm = scaler.inverse_transform(centroides)


In [ ]:
df_centroides = pd.DataFrame(centroides_norm, columns=features_names)
df_centroides

,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,...,CAP_ENDURE_RASTA,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA
0,4.299342,42.821246,77.968948,63176.132836,10.220940,0.060048,0.020786,0.0,0.002310,0.00231,...,0.108516,0.187073,0.032334,0.004619,0.000000,0.981524,0.468837,0.004619,0.856840,0.205549
1,3.383166,42.700755,79.043998,59801.581947,6.659453,0.040496,0.000000,0.0,0.014018,0.00000,...,0.154261,0.911156,0.028036,0.000000,0.023363,0.911221,0.093452,0.000000,0.887793,0.322409


### 1. Imporatanci de las caracteristicas segun la varianza de las variables

In [ ]:
#

import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px

'''
Esta funcion  permite obtener las Features mas importantes de acuerdo
con la varianza de cada caracteristica
'''
def calculo_featrures_importantes(dataframe):


    # Crear un nuevo DataFrame con la desviación estándar
    df_desv_std = df_centroides.std().sort_values(ascending=False).reset_index(drop=False).rename(columns={0: "Std", 'index':'Features'})

    return df_desv_std


'''
Esta funcion permite hacer un grafico de lineas paralelas
de acuerdo a las caracteristicas mas relevantes

Entrada: Dataframe [0:x] con las caracteristicas mas relevnates para analizar
'''
def graficoLineasParalelas(dataframe, columnas_graficar):
    fig = px.parallel_coordinates(dataframe, color="Cluster",
                                dimensions=columnas_graficar ,
                                color_continuous_scale=px.colors.sequential.Viridis,
                                color_continuous_midpoint=0.5)
    # Configurar el layout
    fig.update_layout(
        xaxis_title='Executions Number',
        yaxis_title='Adjusted R2 value',
        legend_title='Metahuristics',
        template='plotly_white',  # Cambiar el diseño
        font=dict(
            family="Arial, monospace",
            size=14,
            color="Black"
        )
    )

    fig.update_layout({"plot_bgcolor": "rgba(0, 0, 0, 0)","paper_bgcolor": "rgba(0, 0, 0, 0)",})

    # Mostrar el gráfico
    fig.show()

'''
    Esta funcion permite seleccionar las caracteristicas mas importantes a graficar
    Ademas permite definir las etiquiteas a mostrar
    '''
def selection_var_graficar(dataframe, n_var):
    df_graficar = dataframe[0:n_var]
    col_graficar= dataframe[0:n_var].index
    return df_graficar, col_graficar

In [ ]:
# Agregamos Grupo a cada Cluster
df_centroides = df_centroides.reset_index(drop=False).rename(columns={"index": "Cluster"})
df_centroides

,Cluster,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,...,CAP_ENDURE_RASTA,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA
0,0,4.299342,42.821246,77.968948,63176.132836,10.220940,0.060048,0.020786,0.0,0.002310,...,0.108516,0.187073,0.032334,0.004619,0.000000,0.981524,0.468837,0.004619,0.856840,0.205549
1,1,3.383166,42.700755,79.043998,59801.581947,6.659453,0.040496,0.000000,0.0,0.014018,...,0.154261,0.911156,0.028036,0.000000,0.023363,0.911221,0.093452,0.000000,0.887793,0.322409


In [ ]:
# Del dataframe centroides selecciona carcateristicas con mayor varianza
# ======================================================================
df_centroides_importantes = calculo_featrures_importantes(df_centroides)
df_centroides_importantes.head(50)

,Features,Std
0,POBLACION_20DIAS_AJT,2386.167817
1,Sol_Ener_Accu_For,112.395674
2,Porc_BLANDO,53.160430
3,Sol_Ener_Accu_Veg,28.887693
4,Porc_FIRME,26.318042
5,Porc_FRIABLE,25.942623
6,Rain_Accu_For,18.446278
7,Rain_Accu_Veg,14.742746
8,Rain_Accu_Mad,13.048426
9,PROFUND_MOTEADOS_RASTA,11.988941


### 1. Analisis de los coeficientes de cada modelo

### 1.1 Analisis Coeficientes Variables Clima
En este caso se analizan los coeficeintes de las variables , relacionadas con el clima. De acuedo a al importancia que se obtuvo en ambos clusters.

In [ ]:
# Variable Clima (5 mas relevantes)
# ================================================================================================================
#select_var_clima =["Rhum_Avg_For","Sol_Ener_Accu_Veg","Rain_Accu_Veg","Rhum_Avg_Veg","Temp_Max_34_Freq_For","Cluster"]
select_var_clima =["Sol_Ener_Accu_For","Sol_Ener_Accu_Veg","Rain_Accu_For","Rain_Accu_Veg","Rain_Accu_Mad","Cluster"]

graficoLineasParalelas(df_centroides[select_var_clima], select_var_clima[0:-1])


### 1.2 Analisis Coeficeintes Variables del Suelo

In [ ]:
# Variable Suelo  (5 mas relevantes)
# ================================================================================================================
#select_var_suelo =["Rhum_Avg_For","Sol_Ener_Accu_Veg","Rain_Accu_Veg","Rhum_Avg_Veg","Temp_Max_34_Freq_For","Cluster"]
select_var_suelo =["Porc_BLANDO","Porc_FRIABLE","Porc_F","PROFUND_RAICES_VIVAS_RASTA", "Porc_FrL", "Cluster"]

graficoLineasParalelas(df_centroides[select_var_suelo], select_var_suelo[0:-1])

###1.3 Analisis  Coeficientes Variables Manejo Agricola.

In [ ]:
# ================================================================================================================
select_var_manejo =["TotN_Emer_Flor","TotK_Emer_Flor", "TotN_Siem_Emer","TotK_Siem_Emer", "TotP_Emer_Flor", "Cluster"]

graficoLineasParalelas(df_centroides[select_var_manejo], select_var_manejo[0:-1])

In [ ]:
features_analize

['Rhum_Avg_For',
 'Sol_Ener_Accu_Veg',
 'ContMalMec_Emer_Flor',
 'Rain_Accu_Veg',
 'Rain_10_Freq_Veg',
 'Rhum_Avg_Veg',
 'Temp_Max_34_Freq_For',
 'Diurnal_Range_Avg_Veg',
 'POSICION_PERFIL_RASTA_PLANO',
 'Rain_Accu_For']

In [ ]:
# Teniendo en cuenta lo de Feature importance muestro Top 10 de las caractersiticas mas relevantes de todo tipo
# ==============================================================================================================
features_Importance =['Rhum_Avg_For',
 'Sol_Ener_Accu_Veg',
 'ContMalMec_Emer_Flor',
 'Rain_Accu_Veg',
 'Rain_10_Freq_Veg',
 'Rhum_Avg_Veg',
 'Temp_Max_34_Freq_For',
 'Diurnal_Range_Avg_Veg',
 'POSICION_PERFIL_RASTA_PLANO',
 'Rain_Accu_For',"Cluster"]

graficoLineasParalelas(df_centroides[features_Importance],features_Importance[0:-1])

## 1. Pruebas Algortimo SpectralClustering
- se varia el numero de componenetes y la semilla de cluster.

In [ ]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("DatasetFinal.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING
# ==============================================================

n_clusters = 0
semilla = 0
best_r2 = 0
r2_train = 0
shiluet_score = 0


for r in range(31):
  dataset_train_cluster = d_train_x
  sc = SpectralClustering(n_clusters=2,assign_labels='discretize',random_state=r).fit(dataset_train_cluster)
  target = sc.labels_
  silhouette_spectral = silhouette_score(dataset_train_cluster, target)
  print("----", silhouette_spectral)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 > r2_train:
    r2_train = r2

  if silhouette_spectral > shiluet_score:
    shiluet_score = silhouette_spectral





  # Con conjunto datos jamas visto  (Conjunto Validación)
  # =======================================================
  grupo_asignado = model_clasf.predict(x_dataset_test)
  y_pred_val = predictionYield(x_dataset_test,models,grupo_asignado)
  r2_new= metricasModelosRegresion(y_dataset_test, y_pred_val)
  print("r2_new: ", r2_new)
  if r2_new > best_r2:
    best_r2 = r2_new
    semilla = r

print("Mejor R2 Traing: ", r2_train)
print("Mejor R2: ", best_r2)
print("Semilla Cluster: ", semilla)
print("Mejor Silhouette: ", shiluet_score)


<ipython-input-2-777a7231a35e>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
---- 0.11926297466733002
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7419823265667128, 0.8577009328173882]


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(


Precisión del modelo clasificación: 96.15%
r2:  0.7383062047798524
r2_new:  0.7561751373539404
---- 0.11926297466733002
Grupos Definitivos:  2


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(


...

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(


r2:  0.7411540368368097
r2_new:  0.7561751373539404
---- 0.11926297466733002
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8577009328173882, 0.7419823265667128]
Precisión del modelo clasificación: 96.15%


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(


r2:  0.7262123913506693
r2_new:  0.7561751373539404
---- 0.11926297466733002
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8577009328173882, 0.7419823265667128]
Precisión del modelo clasificación: 96.92%
r2:  0.7290602234076264
r2_new:  0.7561751373539404
Mejor R2 Traing:  0.7411540368368097
Mejor R2:  0.7561751373539404
Semilla Cluster:  0
Mejor Silhouette:  0.11926297466733002


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.578e+07, tolerance: 3.952e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.160e+06, tolerance: 7.775e+04
  model = cd_fast.enet_coordinate_descent(


**Nota**
- Se realizarón pruebas variando el tamaño de los cluster [2-10], sin embargo se observo que el numero  de registro para K > 2 era muy pequeño , los grupos formados quedaban por debajo de 40 observaciones, por ese motivo se fijo k=2 y se varia unicamente la semilla de el algortimo de clustering

## 2. Prueba Algortimo de Mezclas de Gaussianas

In [ ]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("DatasetFinal.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING
# ==============================================================
semilla = 0
best_r2 = 0
r2_train = 0
shiluet_score=0


for r in range(31):
  dataset_train_cluster = d_train_x
  gm = GaussianMixture(n_components=2,covariance_type = 'full', random_state=r).fit(dataset_train_cluster)
  target =  gm.predict(dataset_train_cluster)
  shiluet_gmn = silhouette_score(dataset_train_cluster, target)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 > r2_train:
    r2_train = r2

  if shiluet_gmn > shiluet_score:
    shiluet_score = shiluet_gmn

  # Con conjunto datos jamas visto  (Conjunto Validación)
  # =======================================================
  grupo_asignado = model_clasf.predict(x_dataset_test)
  y_pred_val = predictionYield(x_dataset_test,models,grupo_asignado)
  r2_new= metricasModelosRegresion(y_dataset_test, y_pred_val)
  print("r2_new: ", r2_new)
  if r2_new > best_r2:
    best_r2 = r2_new
    semilla = r

print("Mejor R2: ", best_r2)
print("Mejor R2 Traing: ", r2_train)
print("Semilla Cluster: ", semilla)
print("Mejor Silhouette: ", shiluet_score)

<ipython-input-2-777a7231a35e>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.8556693643700476, 0.7417871863817992]
Precisión del modelo clasificación: 96.92%
r2:  0.7217598636784903
r2_new:  0.7369770836262105


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.566e+07, tolerance: 3.558e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.860e+06, tolerance: 8.203e+04
  model = cd_fast.enet_coordinate_descent(


Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 93.85%


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


...

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


r2_new:  0.7634672747004629
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 93.85%
r2:  0.7214304231774126
r2_new:  0.7631635686510387


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7431691808719083, 0.8513257852244654]
Precisión del modelo clasificación: 93.85%
r2:  0.7214304231774126
r2_new:  0.7631635686510387
Mejor R2:  0.7660456074517756
Mejor R2 Traing:  0.7232360137963507
Semilla Cluster:  3
Mejor Silhouette:  0.12159914202866326


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.014e+06, tolerance: 8.148e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.577e+07, tolerance: 3.635e+04
  model = cd_fast.enet_coordinate_descent(


## 3.Prueba Algortimo K-MEANS

In [ ]:
#1. Cargue de los conjuntos de datos
#==============================================
df = pd.read_csv("DatasetFinal.csv")
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas (Codificación dummy)
#   Solo a categoricas
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

#3. Division datos (Variable dependinte e independiente)
# ============================================================
dataset_target = dataset.RDT_AJUSTADO
dataset_features = dataset.drop(["ID_LOTE"], axis=1)

# 4. Normalización del datset
# =================================================================
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(dataset_features), columns=dataset_features.columns)


# Division de los datos en Training y Test   (PARTE 1)
# ================================================================
dataset_train, dataset_test = train_test_split(X_normalized, test_size = 0.1, random_state=92)
lista_indices_train = list(dataset_train.index)
lista_indices_test = list(dataset_test.index)


dataset_tag_test = dataset_features.loc[lista_indices_test]
y_dataset_test = dataset_tag_test.RDT_AJUSTADO
x_dataset_test = dataset_tag_test.drop(["RDT_AJUSTADO"],axis=1)


# Division de los datos en Training y Test   (PARTE 2)
# ================================================================
d_train , d_test = train_test_split(dataset_train, test_size = 0.1, random_state=87)
lista_indices_train_p1 = list(d_train.index)
lista_indices_test_p1 = list(d_test.index)
d_train_y = d_train.RDT_AJUSTADO
d_train_x = d_train.drop(["RDT_AJUSTADO"],axis=1)

d_test_y = dataset_features.loc[lista_indices_test_p1].RDT_AJUSTADO
d_test_x = dataset_features.loc[lista_indices_test_p1].drop(["RDT_AJUSTADO"],axis=1)


# CLUSTERING
# ==============================================================
semilla = 0
best_r2 = 0
r2_train = 0
shiluet_score=0

for r in range(31):
  dataset_train_cluster = d_train_x

  target = AlgortimoClustering(dataset_train_cluster, 3,r)
  shiluet_km = silhouette_score(dataset_train_cluster, target)
  dataset_tag = dataset_features.loc[lista_indices_train_p1]
  dataset_tag["Grupo"] =target
  grupos_definitivos  = calcularGruposDefinitivos(dataset_tag)
  print("Grupos Definitivos: ", len(grupos_definitivos))
  # Calculos correlaciones OLMS y modelos
  lista_corr_olsms, models = CalcularCorrelationInitialOLS(grupos_definitivos)
  print("lista Correlaciones Iniciales: ", lista_corr_olsms)
  # Etapa de Clasificación
  model_clasf = etapaClasfDT(dataset_tag)
  # Se valida con lo que nunca se vio.
  grupo_asignado = model_clasf.predict(d_test_x)
  y_pred_test = predictionYield(d_test_x,models,grupo_asignado)
  r2= metricasModelosRegresion(d_test_y, y_pred_test)
  print("r2: ", r2)
  if r2 > r2_train:
    r2_train = r2

  if shiluet_km > shiluet_score:
    shiluet_score = shiluet_km


  # Con conjunto datos jamas visto  (Conjunto Validación)
  # =======================================================
  grupo_asignado = model_clasf.predict(x_dataset_test)
  y_pred_val = predictionYield(x_dataset_test,models,grupo_asignado)
  r2_new= metricasModelosRegresion(y_dataset_test, y_pred_val)
  print("r2_new: ", r2_new)
  if r2_new > best_r2:
    best_r2 = r2_new
    semilla = r

print("Mejor R2: ", best_r2)
print("Mejor R2 Traing: ", r2_train)
print("Semilla Cluster: ", semilla)
print("Mejor Silhouette: ", shiluet_score)

<ipython-input-2-777a7231a35e>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Grupos Definitivos:  2


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.374e+07, tolerance: 8.696e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.245e+07, tolerance: 3.208e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.7509033517752088, 0.8555929156567287]
Precisión del modelo clasificación: 94.62%
r2:  0.6296481821332554
r2_new:  0.40981180222485647
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7507589252458368, 0.8556769421746893]


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(


...

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.8556769421746893, 0.7507589252458368]
Precisión del modelo clasificación: 96.15%
r2:  0.6141065239308785
r2_new:  0.7373825005839272
Grupos Definitivos:  2


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(


lista Correlaciones Iniciales:  [0.8556769421746893, 0.7507589252458368]
Precisión del modelo clasificación: 96.15%
r2:  0.4425741114391827
r2_new:  0.7292293809543291
Grupos Definitivos:  2
lista Correlaciones Iniciales:  [0.7507589252458368, 0.8556769421746893]
Precisión del modelo clasificación: 97.69%
r2:  0.6227392089839975
r2_new:  0.4099244364086919
Mejor R2:  0.742633250153431
Mejor R2 Traing:  0.7074330992732479
Semilla Cluster:  11
Mejor Silhouette:  0.12433274995496106


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.352e+07, tolerance: 8.689e+04
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.444e+07, tolerance: 3.211e+04
  model = cd_fast.enet_coordinate_descent(
